# 03 · From PyTorch to SpiNNaker2: the direction task, trained and transferred

In notebook 02 you built a direction detector by hand. Here a small network learns the same task in snnTorch (spiking layers for PyTorch), is exported to NIR and converted into py-spinnaker2 populations and projections, the objects you built yourself. Then you check, spike for spike, that the converted network computes exactly what the trained one does. Runs on Brian2; cluster participants also run it on the chip (section 7). "After the workshop" at the end holds more for anyone who wants to go further.

▶ marks optional background. Adapted in parts from the vendor's [tutorial 08, "The NIR workflow"](https://gitlab.com/spinnaker2/py-spinnaker2/-/blob/v0.8.5/tutorials/deep_dive/08_nir.ipynb) (SpiNNcloud, Apache-2.0).

## 0. Setup

In [ ]:
# --- libraries ---
import contextlib, io  # to silence the Brian2 progress printout
import numpy as np  # arrays and numbers
import matplotlib.pyplot as plt  # figures
from matplotlib.animation import FuncAnimation, PillowWriter  # animations, saved as GIFs
from IPython.display import Image, display  # shows the GIFs in the notebook
import torch  # PyTorch: tensors and training
import torch.nn as nn  # layers (nn.Linear holds the weights)
import torch.nn.functional as F  # the loss function
import snntorch as snn  # spiking layers for PyTorch (snn.Leaky is the LIF neuron)
from snntorch import surrogate, utils  # surrogate gradients for training; utils.reset clears the potentials
from snntorch.export_nir import export_to_nir  # snnTorch -> NIR
from spinnaker2 import brian2_sim, s2_nir  # py-spinnaker2: the Brian2 backend and the NIR import

# --- backend: where the networks run ---
brian2 = brian2_sim.Brian2Backend()  # the simulator on this computer; the chip run is in section 7
torch.manual_seed(0)  # the same training result on every run

# --- figure style ---
plt.rcParams.update({"axes.grid": True, "axes.axisbelow": True, "grid.color": "0.88", "grid.linewidth": 0.6})  # light grey grid in all figures

### Helpers for figures and animations

Functions that only draw; nothing about the network.

In [ ]:
def animate_examples(rows, n_pixels, filename, steps, trail=True):
    """Save and show a GIF of a grid of inputs {row label: {title: {pixel: [time steps]}}}: one strip of pixels per example, all in sync."""
    n_rows, n_cols = len(rows), max(len(inputs) for inputs in rows.values())
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 0.9 * n_rows + 0.8), squeeze=False)
    strips = []  # [image, spikes, trace] per example
    for r, (label, inputs) in enumerate(rows.items()):
        for c, ax in enumerate(axes[r]):
            ax.set(xticks=[], yticks=[])
            ax.grid(False)
            if c >= len(inputs):
                ax.axis("off")
                continue
            title, spikes = list(inputs.items())[c]
            strips.append([ax.imshow(np.zeros((1, n_pixels)), cmap="gray_r", vmin=0, vmax=1), spikes, np.zeros(n_pixels)])
            ax.set_title(title, fontsize=9)
        axes[r][0].set_ylabel(label, rotation=0, ha="right", va="center")
    step_text = fig.suptitle("")

    def update(t):
        for strip in strips:
            image, spikes, trace = strip
            fired = np.array([1.0 if t in spikes.get(p, []) else 0.0 for p in range(n_pixels)])
            strip[2] = (0.6 if trail else 0.0) * trace + fired  # older events fade (only for the eye)
            image.set_data(np.clip(strip[2], 0, 1)[None, :])
        step_text.set_text(f"time step {t}")
        return [strip[0] for strip in strips]

    plt.tight_layout()
    FuncAnimation(fig, update, frames=steps, interval=400).save(filename, writer=PillowWriter(fps=2.5))
    plt.close(fig)
    display(Image(filename=filename))


def plot_predictions(results, n_pixels, names):
    """Plot test examples {title: (direction, input spikes, output spikes (time steps, outputs))}: input on top, output spikes and answer below."""
    fig, axes = plt.subplots(2, len(results), figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, squeeze=False)
    for c, (title, (direction, spikes, out)) in enumerate(results.items()):
        ax = axes[0][c]
        ax.eventplot([spikes.get(p, []) for p in range(n_pixels)], lineoffsets=range(n_pixels), linelengths=0.6, colors="black")
        ax.set(title=title, yticks=range(n_pixels), ylim=(-0.6, n_pixels - 0.4))
        counts = out.sum(axis=0).astype(int)
        ax = axes[1][c]
        for k in range(len(names)):
            ax.eventplot([np.nonzero(out[:, k])[0]], lineoffsets=[k], linelengths=0.6, colors=f"C{k}")
        ax.set(yticks=range(len(names)), yticklabels=[f"{name}: {counts[k]}" for k, name in enumerate(names)],
               ylim=(-0.6, len(names) - 0.4), xlim=(-0.5, out.shape[0] - 0.5), xlabel="time step (1 step = 1 ms)")
        answer = names[counts.argmax()] if counts.max() > np.sort(counts)[-2] else "none (tie)"
        correct = answer == direction
        ax.set_title(f"answer: {answer}, {'correct' if correct else 'wrong'}", fontsize=10, color="black" if correct else "C3")
    axes[0][0].set_ylabel("input pixel")
    axes[1][0].set_ylabel("output spikes")
    plt.tight_layout()
    plt.show()

## 1. The task from notebook 02, as a dataset

The same input: a dot moves across four pixels, to the right or to the left, fast (`step=1`) or slowly (`step=2`). To learn, the network needs examples with the right answer (the label: 0 = right, 1 = left). The dot starts at different time steps (2 to 8), so the network cannot simply learn one fixed moment. Two test sets it never sees during training: later start times, and inputs where one pixel misses the dot.

The animations below show the examples: each strip is one example, pixels 0 to 3 from left to right, all running in sync. The training set varies only the direction, the speed and the start time.

In [ ]:
# --- the input, as in notebook 02 ---
N_PIXELS, STEPS = 4, 20  # 4 pixels, 20 time steps per example


def moving_dot(direction, step=1, start=5):
    """Spike times of the pixels, {pixel: [time step]}, for a dot moving "right" or "left" (as in notebook 02)."""
    order = range(N_PIXELS) if direction == "right" else reversed(range(N_PIXELS))
    return {pixel: [start + k * step] for k, pixel in enumerate(order)}


def without(spikes, pixel):
    """The same input with one pixel missing the dot."""
    return {p: ([] if p == pixel else times) for p, times in spikes.items()}


# --- from spike times to PyTorch tensors ---
def to_tensor(spikes):
    """Spike times -> tensor of shape (time steps, pixels) with 1 where a pixel fires."""
    x = torch.zeros(STEPS, N_PIXELS)
    for pixel, times in spikes.items():
        for t in times:
            x[t, pixel] = 1
    return x


def make_set(examples):
    """List of (direction, spikes) -> inputs (time steps, examples, pixels) and labels (0 = right, 1 = left)."""
    x = torch.stack([to_tensor(spikes) for _, spikes in examples], dim=1)
    y = torch.tensor([0 if direction == "right" else 1 for direction, _ in examples])
    return x, y


# --- the datasets: 28 for training, two test sets ---
train_examples = [(d, moving_dot(d, s, t0)) for d in ("right", "left") for s in (1, 2) for t0 in range(2, 9)]
later_examples = [(d, moving_dot(d, s, t0)) for d in ("right", "left") for s in (1, 2) for t0 in range(9, 13)]
missing_examples = [(d, without(moving_dot(d, s), p)) for d in ("right", "left") for s in (1, 2) for p in range(N_PIXELS)]
x_train, y_train = make_set(train_examples)
print(f"{len(train_examples)} training examples, {len(later_examples)} with later start times, {len(missing_examples)} with one pixel missing")

In [ ]:
# --- the examples, animated ---
speed = {1: "fast", 2: "slow"}
train_rows = {f"{d}, {speed[s]}": {f"start {t0}": moving_dot(d, s, t0) for t0 in range(2, 9)}
              for d in ("right", "left") for s in (1, 2)}  # all 28 training examples
print("Train dataset")
animate_examples(train_rows, N_PIXELS, "direction_training_examples.gif", steps=STEPS)

test_rows = {  # 4 examples of each test set
    "later start": {f"{d}, {speed[s]}, start {t0}": moving_dot(d, s, t0) for d, s, t0 in [("right", 1, 9), ("right", 2, 12), ("left", 1, 10), ("left", 2, 11)]},
    "pixel missing": {f"{d}, {speed[s]}, no pixel {p}": without(moving_dot(d, s), p) for d, s, p in [("right", 1, 1), ("right", 2, 2), ("left", 1, 0), ("left", 2, 3)]},
}
print("Test dataset")
animate_examples(test_rows, N_PIXELS, "direction_test_examples.gif", steps=STEPS)

## 2. The network in snnTorch

Four input pixels, a hidden layer of 8 LIF neurons and two outputs, one per direction; the answer is the output that fires more often. `nn.Linear` holds the weights (like `W` in notebook 02), `snn.Leaky` is a layer of LIF neurons (`beta` is `alpha_decay`). There are no delays: the network has to read the order of the spikes with its weights alone.

How `beta` turns into `alpha_decay` when the network is converted for the chip is explained in section 5, in the collapsible "Why these settings".

<details><summary>PyTorch in a nutshell: how the network below is written</summary>

- **Tensor:** PyTorch's array, like a NumPy array, that also records how it was computed, so that gradients can flow back through it. `x_train` has the shape (time steps, examples, pixels) = (20, 28, 4).
- **Module:** every building block is an `nn.Module`. Calling it on a tensor computes its output: `y = layer(x)`.
- **`nn.Linear(4, 8, bias=False)`:** a weight matrix of shape (8, 4), one row per hidden neuron and one column per pixel, like `W` in notebook 02. It computes `x @ W.T`. The weights start random; they are what training changes (`net[0].weight`).
- **`snn.Leaky(...)`:** snnTorch's layer of LIF neurons, also a module. It has no weights, only `beta`, the threshold and the potential it carries from one call to the next.
- **`nn.Sequential(a, b, c, d)`:** chains modules, so `net(x)` computes `d(c(b(a(x))))`. `net[0]` is the first layer, `net[1]` the second, and so on.
- **Batches:** every layer works on many examples at once; the first dimension of its input is the example. One time step of all examples, `x_t`, has the shape (examples, pixels).
- **Time:** a PyTorch layer has no notion of time. The loop in `run_snntorch` steps through the time steps and calls the layers once per step; `snn.Leaky` keeps the potential between the calls.

`run_snntorch` calls the layers one by one (`net[1](net[0](x_t))`) instead of `net(x_t)` only to keep the hidden spikes for the comparison in section 6. More: [PyTorch tutorial "Build the neural network"](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html).

</details>

<details><summary>The update rule of <code>snn.Leaky</code></summary>

Per time step, `snn.Leaky` computes $U[t+1] = \beta\, U[t] + I_\text{in}[t+1]$, where $I_\text{in}$ is the output of the preceding `nn.Linear`. The neuron spikes when $U$ exceeds the threshold (1 by default); with `reset_mechanism="zero"` it is then set to 0. This is the update of the `lif` neuron of notebook 01, with `beta` in the role of `alpha_decay`. `init_hidden=True` lets each layer keep its potential internally, so `nn.Sequential` works. Source: the `snn.Leaky` docstring and [snnTorch tutorial 3](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_3.html).

</details>

In [ ]:
# --- the network ---
beta = 0.5  # share of the potential kept per step (alpha_decay in py-spinnaker2)
spike_grad = surrogate.fast_sigmoid()  # smooth stand-in for the step function, used only for training
net = nn.Sequential(
    nn.Linear(N_PIXELS, 8, bias=False),  # weights pixels -> hidden
    snn.Leaky(beta=beta, init_hidden=True, reset_mechanism="zero", spike_grad=spike_grad),  # 8 LIF neurons, reset to 0 after a spike
    nn.Linear(8, 2, bias=False),  # weights hidden -> outputs
    snn.Leaky(beta=beta, init_hidden=True, reset_mechanism="zero", spike_grad=spike_grad, output=True),  # 2 LIF outputs; output=True also returns the potential
)
print(net)

### Running the network over time

**In short:** `run_snntorch` plays an input to the network step by step and returns which neurons spiked when; training and the comparison with the converted network both need these spikes.

An input is not a single picture but a short sequence: 20 time steps, and in each step every pixel fires (1) or not (0). A PyTorch layer computes only one step per call, and each LIF neuron keeps its potential from one call to the next. `run_snntorch` therefore feeds the input to the network one time step after the other and records which neurons spike in each step. All examples go through at once: each step holds the pixels of every example.

Section 3 sums the output spikes into counts for the loss; section 6 compares the spikes, hidden layer included, with those of the converted network.

In [ ]:
# --- run the network over all time steps ---
def run_snntorch(x):
    """Run the network step by step over inputs (time steps, examples, pixels); return hidden and output spikes over time."""
    utils.reset(net)  # set all potentials to 0, so nothing is left over from the previous input
    hidden, output = [], []  # the spikes of each time step
    for x_t in x:  # one time step: the pixels of all examples, shape (examples, pixels)
        h = net[1](net[0](x_t))  # weights, then the 8 hidden LIF neurons: which of them spike now (0 or 1)
        o, _ = net[3](net[2](h))  # weights, then the 2 output LIF neurons: their spikes (the potentials are not used)
        hidden.append(h)  # the hidden spikes are kept for the comparison in section 6
        output.append(o)
    return torch.stack(hidden), torch.stack(output)  # lists of steps -> tensors (time steps, examples, neurons)

**Go deeper:** new to training spiking networks? The snnTorch tutorials build it up step by step, with the same time loop as `run_snntorch`: [tutorial 3](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_3.html) (a layer of LIF neurons, run over time), [tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html) (training with backpropagation through time and a loss on the output spikes), [tutorial 6](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_6.html) (surrogate gradients). The overview behind them: [Eshraghian et al. 2023, "Training Spiking Neural Networks Using Lessons From Deep Learning"](https://doi.org/10.1109/JPROC.2023.3308088). Workshop note: [Spiking Neurons and Networks, section 8](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).

## 3. Train it

The loss is the cross-entropy of the two spike counts, as in a classifier, and the surrogate gradient stands in for the spike's missing gradient (slides before this block). With 28 examples, training takes a few seconds.

In [ ]:
# --- training: 300 passes over the 28 examples ---
optimizer = torch.optim.Adam(net.parameters(), lr=0.05)
for epoch in range(1, 301):
    counts = run_snntorch(x_train)[1].sum(dim=0)  # spike count of each output neuron
    loss = F.cross_entropy(counts, y_train)
    optimizer.zero_grad()
    loss.backward()  # backpropagation through time, with the surrogate gradient
    optimizer.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch}: loss {loss.item():.3f}")


# --- accuracy on the three sets ---
def accuracy(examples):
    """Share of examples for which the right output neuron fires more often (a tie counts as wrong)."""
    x, y = make_set(examples)
    with torch.no_grad():
        counts = run_snntorch(x)[1].sum(dim=0)
    return ((counts.argmax(dim=1) == y) & (counts[:, 0] != counts[:, 1])).float().mean().item()


for name, examples in [("training", train_examples), ("later start times", later_examples), ("one pixel missing", missing_examples)]:
    print(f"{name:18s}: {accuracy(examples):.0%} correct")

### Three test examples

Three inputs the network never saw during training: a later start, a missing pixel it still gets right, and a missing pixel it gets wrong. Top: the input pixels. Bottom: the spikes of the two output neurons and their counts; the answer is the output with more spikes.

In [ ]:
# --- three test examples: input, output spikes, answer ---
cases = {
    "later start\n(right, slow, start 12)": ("right", moving_dot("right", step=2, start=12)),
    "pixel 0 missing\n(left, fast)": ("left", without(moving_dot("left"), 0)),
    "pixel 3 missing\n(left, fast)": ("left", without(moving_dot("left"), 3)),
}
results = {}
for title, (direction, spikes) in cases.items():
    with torch.no_grad():
        out = run_snntorch(to_tensor(spikes).unsqueeze(1))[1][:, 0]  # output spikes of this one example, (time steps, 2)
    results[title] = (direction, spikes, out.numpy())
plot_predictions(results, N_PIXELS, ["right", "left"])

## 4. Export to NIR

NIR (Neuromorphic Intermediate Representation) describes the network as a graph of building blocks: weight layers (`Linear`) and neuron groups (`LIF`), connected by edges. snnTorch runs the model once on an example input to record this graph; only the size of the input matters, not its values.

<details><summary>What NIR is, and why it exists</summary>

Every spiking framework and chip describes neurons in its own way, so a trained network used to be tied to one of them. NIR is a shared file format, comparable to ONNX for deep learning: each building block (a "primitive") is defined by a continuous-time equation, independent of any simulator's time step. The LIF primitive, for example, is

$$\tau\, \dot v = (v_\text{leak} - v) + R\, I, \qquad \text{spike if } v > v_\text{thr}.$$

Each framework exports to it, each chip imports from it and discretises the equations with its own step. [Pedersen et al. 2024 (Nature Communications)](https://doi.org/10.1038/s41467-024-52259-9) introduce NIR and show it with 7 simulators and 4 hardware platforms, SpiNNaker2 among them. The list of primitives: [NIR documentation](https://neuroir.org/docs/primitives/).

</details>

In [ ]:
nir_graph = export_to_nir(net, torch.zeros(N_PIXELS))  # runs the model once on a dummy input to record the graph
nir_graph.infer_types()  # fill in the input and output shapes of every node
s2_nir.model_summary(nir_graph)  # one line per node: input, Linear, LIF, Linear, LIF, output

## 5. Convert it into a py-spinnaker2 network

`s2_nir.from_nir` turns the NIR graph into `snn.Population` and `snn.Projection` objects, the same objects you built by hand in notebook 02. A `ConversionConfig` controls the translation. **Rule for models from snnTorch:** `dt = 1e-4`, integrator `FORWARD`, and the reset rule of the snnTorch model. A wrong setting gives no error message, only different spikes (section 6 checks them).

<details><summary>Why these settings</summary>

| Setting | Value for snnTorch | Why |
|---|---|---|
| `dt` | `1e-4` | snnTorch stores `beta` as a time constant $\tau = \Delta t / (1 - \beta)$ with a fixed $\Delta t = 10^{-4}$ s; only the same `dt` gives back `beta` |
| `integrator` | `FORWARD` | snnTorch computes $v[t+1] = \beta\, v[t] + \text{input}$, a forward Euler step |
| `reset` | `ZERO` here | NIR does not store the reset rule, so it has to be set by hand to match `reset_mechanism` |

In numbers, for this network's $\beta = 0.5$, so $\tau = 2 \cdot 10^{-4}$ s, py-spinnaker2 turns $\tau$ back into `alpha_decay`, the decay factor per step:

- `FORWARD` ([forward Euler](https://en.wikipedia.org/wiki/Euler_method)): $\alpha = 1 - \Delta t / \tau = 0.5$, exactly snnTorch's `beta`
- `EXPONENTIAL` (the default, the exact solution of the decay): $\alpha = e^{-\Delta t / \tau} = e^{-0.5} \approx 0.607$

Both are valid ways to discretise the NIR equation; only the forward one reproduces what snnTorch computed. The smaller `beta`, the larger the difference: for $\beta = 0.9$ it would be 0.905 against 0.9, often too small to change a spike. That is why this network uses $\beta = 0.5$. A different `dt` changes $\alpha$ as well: with `dt = 1e-3`, forward Euler gives $\alpha = 0$ for this network, a neuron without memory. And $\beta = 1$ (no leak) gives $\tau = \infty$, which the conversion cannot handle: the threshold becomes `nan`, again without an error.

A fourth difference concerns the comparison, not the settings: the chip delays every spike by one step per projection, snnTorch does not (section 6).

Sources: `convert_LIF` in py-spinnaker2's `s2_nir.py`; [py-spinnaker2 API docs, s2_nir](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/s2_nir-api.html); the workshop note [Spiking Neurons and Networks, section 8](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).

</details>

<details><summary>What <code>scale_weights</code> does</summary>

NIR layers become `lif_no_delay` populations on the chip, with 8-bit integer weights (−127…127) and no synaptic delays (hence `conn_delay = 0`). `scale_weights = True` multiplies the weights of each layer so that the largest one becomes 127, and multiplies the thresholds by the same factor, so the network behaves the same apart from rounding. With `weight_scale_percentile` below 100, a percentile of the weights is mapped to 127 instead, which helps when a few outliers would otherwise squeeze all other weights into a few integer values. Source: [py-spinnaker2 API docs, s2_nir](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/s2_nir-api.html).

</details>

In [ ]:
# --- the conversion settings; the defaults are the rule for snnTorch models ---
def make_config(integrator=s2_nir.IntegratorMethod.FORWARD, reset=s2_nir.ResetMethod.ZERO):
    cfg = s2_nir.ConversionConfig()
    cfg.dt = 1e-4  # the rule for snnTorch models
    cfg.integrator = integrator
    cfg.reset = reset
    cfg.output_record = ["spikes"]  # record the spikes of the output layer
    cfg.conn_delay = 0  # lif_no_delay has no synaptic delays
    cfg.scale_weights = True  # scale the float weights into the chip's integer range
    return cfg


# --- convert and look at the result ---
s2_net, inputs, outputs = s2_nir.from_nir(nir_graph, make_config())  # the network, its input and output populations
for pop in s2_net.populations:
    print(f"population {pop.name}: {pop.size} neurons, model {pop.app_class.app_name}")
first = [p for p in s2_net.projections if p.pre in inputs][0]  # the projection pixels -> hidden
print(f"\nprojection {first.pre.name} -> {first.post.name}, weights as a matrix (one row per hidden neuron, one column per pixel, like W in notebook 02):")
print(first.sparse_weights.toarray().T.astype(int))
print("the same weights in snnTorch:")
print(np.round(net[0].weight.detach().numpy(), 2))

These are the objects of notebook 02: a `spike_list` population for the pixels, two `lif_no_delay` populations and two projections, the first printed above as a matrix like `W`. New: the weights are whole numbers from −127 to 127, scaled so that the largest trained weight becomes 127, and the threshold is scaled by the same factor. The range is wider than the −15 to 15 of `lif` because `lif_no_delay` has no delays: the bits that `lif` uses for the delay hold the weight instead. Rounding can change a spike when a potential lands very close to the threshold; section 6 checks whether it does here.

## 6. Same network? Compare spike for spike

Run one input through the converted network on Brian2 and through the original in snnTorch, and compare every spike of the hidden and the output layer. On SpiNNaker2 each spike reaches the next layer one step later than in snnTorch, so the py-spinnaker2 spikes are shifted back by one step for the hidden layer and two for the output layer before comparing.

<details><summary>Why one step per projection</summary>

In snnTorch, a spike computed in step $t$ reaches the next layer in the same step $t$, because the whole forward pass runs within one step. On the chip, each layer runs on its own cores and a spike travels to the next core as a packet, which is processed in the following step (the `delay + 1` rule of hands-on 1, with `delay = 0`). A network with $L$ projections therefore produces its output $L$ steps later. For classification by spike count, this hardly matters; for spike-for-spike comparisons and for networks that react in real time, it does.

</details>

In [ ]:
# --- helpers: spikes of both versions, and the comparison ---
def spikes_py_spinnaker2(spikes, cfg, backend=brian2):
    """Convert with cfg, run one input, return hidden and output spikes, shifted back by one step per projection."""
    s2_net, inputs, outputs = s2_nir.from_nir(nir_graph, cfg)
    inputs[0].params = spikes  # the input population sends this input
    hidden = [p for p in s2_net.populations if p not in inputs and p not in outputs][0]
    hidden.record = ["spikes"]  # also record the hidden layer
    quiet = isinstance(backend, brian2_sim.Brian2Backend)  # Brian2 prints two progress lines per run; the chip's printout stays
    with contextlib.redirect_stdout(io.StringIO()) if quiet else contextlib.nullcontext():
        backend.run(s2_net, STEPS + 2)  # 2 extra steps for the delay of the two projections
    shift = lambda d, k: {i: [int(t) - k for t in ts] for i, ts in d.items() if len(ts)}  # undo k steps of delay
    return shift(hidden.get_spikes(), 1), shift(outputs[0].get_spikes(), 2)


def spikes_snntorch(spikes):
    """Hidden and output spike times of the original snnTorch model for one input."""
    with torch.no_grad():
        hidden, output = run_snntorch(to_tensor(spikes).unsqueeze(1))
    as_dict = lambda m: {i: torch.nonzero(m[:, 0, i]).flatten().tolist() for i in range(m.shape[2]) if m[:, 0, i].any()}
    return as_dict(hidden), as_dict(output)


def compare(cfg, label, examples):
    """Count the inputs for which hidden and output spikes are identical, and the right answers of the converted network."""
    same_hidden = same_output = right = 0
    for direction, spikes in examples:
        s2_h, s2_o = spikes_py_spinnaker2(spikes, cfg)
        t_h, t_o = spikes_snntorch(spikes)
        same_hidden += s2_h == t_h
        same_output += s2_o == t_o
        counts = [len(s2_o.get(i, [])) for i in range(2)]
        right += counts[0] != counts[1] and int(np.argmax(counts)) == (0 if direction == "right" else 1)
    n = len(examples)
    print(f"{label}: hidden layer identical in {same_hidden}/{n}, output layer in {same_output}/{n}; right answers {right}/{n}")


def plot_spikes(runs):
    """Rasters of hidden (grey) and output (orange) spikes, one panel per run {title: (hidden, output)}."""
    fig, axes = plt.subplots(1, len(runs), figsize=(10, 4), sharey=True)
    for ax, (title, (h, o)) in zip(axes, runs.items()):
        for i, ts in h.items():
            ax.plot(ts, [i] * len(ts), "|", color="0.4", ms=12, mew=2)
        for i, ts in o.items():
            ax.plot(ts, [9 + i] * len(ts), "|", color="C1", ms=12, mew=2)
        ax.set(title=title, xlabel="time step", xlim=(0, STEPS), yticks=list(range(8)) + [9, 10],
               yticklabels=[f"hidden {i}" for i in range(8)] + ["out: right", "out: left"])
    plt.tight_layout()
    plt.show()

In [ ]:
# --- one input, spike for spike ---
one_input = moving_dot("right")
s2_h, s2_o = spikes_py_spinnaker2(one_input, make_config())
t_h, t_o = spikes_snntorch(one_input)
plot_spikes({"snnTorch": (t_h, t_o), "py-spinnaker2 on Brian2 (shifted back)": (s2_h, s2_o)})

# --- all 60 inputs ---
all_examples = train_examples + later_examples + missing_examples
compare(make_config(), "correct settings", all_examples)

## 7. On the SpiNNaker2 chip (cluster participants)

Set `USE_HARDWARE = True` and run the cell: the converted network runs for one input on one 48-chip board (about one minute, see notebook 02, section 4, for the setup) and is compared spike for spike with snnTorch. Without cluster access, the cell runs the same check on Brian2.

In [ ]:
# --- backend ---
USE_HARDWARE = False  # True: a SpiNNaker2 board (HPI cluster only); False: Brian2, as a dry run

if USE_HARDWARE:
    from spinnaker2 import hardware  # the chip backend, only needed on the cluster

    hw = hardware.SpiNNcloud48NodeBoard()  # one 48-chip board, reserved for this run
else:
    hw = brian2_sim.Brian2Backend()
print("Running on backend:", type(hw).__name__)

# --- run one input and compare with snnTorch ---
chip_h, chip_o = spikes_py_spinnaker2(moving_dot("left"), make_config(), backend=hw)
t_h, t_o = spikes_snntorch(moving_dot("left"))
print(f"{type(hw).__name__}: hidden layer {'identical' if chip_h == t_h else 'DIFFERENT'}, output layer {'identical' if chip_o == t_o else 'DIFFERENT'} to snnTorch")

---

**In short:** the task of notebook 02, solved by training instead of by hand: trained in snnTorch, exported to NIR and converted into populations and projections with integer weights. With the rule for snnTorch models (`dt = 1e-4`, `FORWARD`, the right reset), the converted network fires exactly the same spikes, on Brian2 and on the chip, shifted by one step per projection. A wrong setting would change the spikes without any warning, so a converted network is worth checking spike for spike. Notebook 04 scales the task up: an 8 × 8 sensor, four directions, noise and lost events, and a network that has to generalise.

**Go deeper:** tutorial 08, sections 2 to 4 (the NIR primitives py-spinnaker2 supports, the same sanity check, settings for large trained models); [py-spinnaker2 user guide, NIR](https://spinnaker2.gitlab.io/py-spinnaker2/user_guide/qs-nir.html); workshop notes: [the conversion rule for snnTorch models](../01_knowledge_base/01_primer_snn_basics.md#8-snns-and-deep-learning).

---

## After the workshop: more to explore

Not part of the workshop time. Two additions for anyone who wants to go further; each needs only sections 0 to 6 to have run.

### A. Break it on purpose

Neither of the following mistakes raises an error: py-spinnaker2 converts and runs the network either way. The cell below converts the network twice with one wrong setting each and compares it with snnTorch, as in section 6: once with the py-spinnaker2 default integrator `EXPONENTIAL`, once with the reset rule `SUBTRACT` instead of `ZERO`.

**Try:** predict first: will the hidden spikes change? The output spikes? The answers? Then run the cell.

**Question:** which of these mistakes would you notice in a classifier that only counts output spikes, and which only in a spike-for-spike comparison? What does that mean for checking a converted network?

In [ ]:
compare(make_config(integrator=s2_nir.IntegratorMethod.EXPONENTIAL), "integrator EXPONENTIAL", all_examples)
compare(make_config(reset=s2_nir.ResetMethod.SUBTRACT), "reset SUBTRACT", all_examples)

*Your answer:*

- …

### B. Look inside: how does it tell the direction without delays?

The hand-built detector of notebook 02 delayed the early spikes so that all four arrived together for one direction. The trained network has no delays, only weights. The cell below prints the integer weights from the pixels to the hidden neurons again, and which hidden neurons fire, and when, for a dot moving right or left, fast and slowly.

**Try:** find a hidden neuron that fires for one direction only. Which pixels excite it (positive weight), which inhibit it (negative weight), and why does it stay silent when the dot moves the other way? (`beta = 0.5`: the potential halves in every step.)

In [ ]:
w_int = first.sparse_weights.toarray().T.astype(int)  # pixels -> hidden, from section 5
print("integer weights from pixels 0, 1, 2, 3:")
for i, row in enumerate(w_int):
    print(f"  hidden {i}: {row}")

print("\nhidden neurons that fire, {neuron: [time steps]} (snnTorch):")
for direction in ["right", "left"]:
    for step in (1, 2):
        hidden, output = spikes_snntorch(moving_dot(direction, step))
        print(f"  dot moving {direction:5s}, step={step}: {hidden}")

*Your answer:*

- …

<details><summary>What the trained network does</summary>

Look for hidden neurons with positive weights from the pixels the dot reaches first in one direction and strongly negative weights from the pixels it reaches last. Moving in that direction, the first pixel lifts the neuron above the threshold at once, before any inhibition arrives. Moving the other way, the inhibiting pixels fire first and push the potential far below zero, so the exciting pixels can no longer lift it to the threshold: the inhibition vetoes the spike. Other neurons do the mirror image for the other direction. Instead of delaying spikes until they arrive together, the trained network uses which pixel fires first, with inhibition from the neighbours as a veto, a mechanism also proposed for direction-selective cells in the retina ([Barlow and Levick 1965](https://doi.org/10.1113/jphysiol.1965.sp007638)). The veto fades with the leak: for a slow dot, the inhibition may have decayed by the time the exciting pixels fire, and the neuron fires for the wrong direction too. Like the hand-built detector, the trained one has a speed limit.

In our run (seed 0), hidden 5 is such a neuron for a dot moving right (weights 42, 35, −127, −94 from pixels 0 to 3; for a slow dot moving left it fires anyway at time step 11), and hidden 0, 2 and 7 do the mirror image. Other versions of PyTorch or snnTorch can give other numbers and other neurons.

</details>